# LAB | Feature Engineering

Notebook resuelto en castellano siguiendo la estructura del lab y los conceptos vistos en clase: revisión inicial, tratamiento de nulos, transformación de variables, dummies, matriz de correlación, test de asociación, train/test split y modelo KNN.

## 1. Cargar los datos

Trabajamos con el dataset **Spaceship Titanic**. Primero importamos las librerías necesarias y cargamos el archivo `spaceship_titanic.csv`.

In [ ]:
# Librerías
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import chi2_contingency

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
# Cargar el dataset
spaceship = pd.read_csv("spaceship_titanic.csv")

# Mostrar las primeras filas
spaceship.head()

## 2. Comprobar la forma del dataset

Revisamos cuántas filas y columnas tiene el dataset.

In [ ]:
# Filas y columnas
spaceship.shape

In [ ]:
print("Número de filas:", spaceship.shape[0])
print("Número de columnas:", spaceship.shape[1])

## 3. Comprobar tipos de datos

Antes de preparar el modelo, revisamos qué columnas son numéricas, categóricas o booleanas.

In [ ]:
# Tipos de datos
spaceship.dtypes

In [ ]:
# Vista general del dataframe
spaceship.info()

## 4. Comprobar valores nulos

Analizamos cuántos valores faltantes hay en cada columna.

In [ ]:
# Valores nulos por columna
spaceship.isna().sum()

In [ ]:
# Porcentaje de valores nulos por columna
(spaceship.isna().mean() * 100).round(2)

## 5. Tratamiento de valores nulos

El enunciado indica que, como hay una cantidad baja de valores nulos, podemos eliminar las filas que contengan valores faltantes.

Esta estrategia es sencilla y válida para este lab, aunque en proyectos reales también podríamos plantear imputaciones según el tipo de variable.

In [ ]:
# Crear una copia limpia eliminando filas con nulos
spaceship_clean = spaceship.dropna().copy()

print("Shape original:", spaceship.shape)
print("Shape tras eliminar nulos:", spaceship_clean.shape)
print("Filas eliminadas:", spaceship.shape[0] - spaceship_clean.shape[0])

In [ ]:
# Comprobación final de nulos
spaceship_clean.isna().sum()

## 6. Transformar la columna `Cabin`

La columna `Cabin` es demasiado granular. En lugar de usar toda la cabina, extraemos solo la primera letra, que representa la cubierta:

`A`, `B`, `C`, `D`, `E`, `F`, `G` o `T`.

In [ ]:
# Ver algunos valores originales de Cabin
spaceship_clean["Cabin"].head()

In [ ]:
# Extraer la primera letra de Cabin
spaceship_clean["Cabin"] = spaceship_clean["Cabin"].str[0]

# Comprobar categorías resultantes
spaceship_clean["Cabin"].value_counts().sort_index()

## 7. Eliminar `PassengerId` y `Name`

`PassengerId` y `Name` son identificadores muy específicos. No aportan valor directo para generalizar el modelo y pueden introducir ruido, por eso los eliminamos.

In [ ]:
# Eliminar columnas identificadoras
spaceship_clean = spaceship_clean.drop(columns=["PassengerId", "Name"])

spaceship_clean.head()

## 8. Matriz de correlación

Antes de crear dummies, revisamos la correlación entre variables numéricas y la variable objetivo.

Como `Transported` es booleana, la convertimos temporalmente a entero para incluirla en la matriz.

In [ ]:
# Copia para análisis de correlación
corr_df = spaceship_clean.copy()
corr_df["Transported"] = corr_df["Transported"].astype(int)

# Seleccionar columnas numéricas
numeric_df = corr_df.select_dtypes(include=np.number)

# Matriz de correlación
corr_matrix = numeric_df.corr()
corr_matrix

In [ ]:
# Visualización de la matriz de correlación
plt.figure(figsize=(10, 6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Matriz de correlación de variables numéricas")
plt.show()

### Interpretación de la correlación

La matriz permite ver qué variables numéricas están más relacionadas con `Transported`. En este dataset, los gastos en servicios como `RoomService`, `Spa` o `VRDeck` suelen tener una relación negativa con la variable objetivo, mientras que otras variables tienen relaciones más débiles.

Esto ayuda a entender qué variables podrían aportar más información al modelo.

## 9. Test de asociación para variables categóricas

Para las variables categóricas usamos un test Chi-cuadrado. Este test permite comprobar si existe asociación entre una variable categórica y la variable objetivo `Transported`.

Hipótesis:
- **H0:** no hay asociación entre la variable y `Transported`.
- **H1:** sí hay asociación entre la variable y `Transported`.

In [ ]:
# Variables categóricas para el test de asociación
categorical_cols = spaceship_clean.select_dtypes(include=["object", "bool"]).columns.tolist()
categorical_cols = [col for col in categorical_cols if col != "Transported"]

chi2_results = []

for col in categorical_cols:
    contingency_table = pd.crosstab(spaceship_clean[col], spaceship_clean["Transported"])
    chi2, p_value, dof, expected = chi2_contingency(contingency_table)
    chi2_results.append({
        "variable": col,
        "chi2": chi2,
        "p_value": p_value,
        "significativa_0.05": p_value < 0.05
    })

chi2_results_df = pd.DataFrame(chi2_results).sort_values("p_value")
chi2_results_df

### Interpretación del test Chi-cuadrado

Las variables con `p_value < 0.05` muestran una asociación estadísticamente significativa con `Transported`. Esto no significa causalidad, pero sí indica que esas variables pueden ser útiles para el modelo.

## 10. Crear dummies para variables no numéricas

Los modelos de Machine Learning no trabajan directamente con texto. Por eso convertimos las variables categóricas en variables dummy.

In [ ]:
# Crear dummies para variables categóricas
spaceship_encoded = pd.get_dummies(spaceship_clean, drop_first=True)

spaceship_encoded.head()

In [ ]:
# Comprobar el nuevo tamaño del dataframe
spaceship_encoded.shape

## 11. Preparar `X` e `y`

Separamos las variables predictoras (`X`) de la variable objetivo (`y`).

In [ ]:
# Variables predictoras y target
X = spaceship_encoded.drop(columns=["Transported"])
y = spaceship_encoded["Transported"]

print("Shape de X:", X.shape)
print("Shape de y:", y.shape)

In [ ]:
# Revisar columnas usadas como predictoras
X.columns

## 12. Train/Test Split

Dividimos el dataset en entrenamiento y test. Usamos el 80% para entrenar y el 20% para evaluar.

Esta separación es fundamental porque permite medir el rendimiento del modelo sobre datos que no ha visto durante el entrenamiento.

In [ ]:
# División train/test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=0
)

print("Dimensión de X_train:", X_train.shape)
print("Dimensión de X_test:", X_test.shape)
print("Dimensión de y_train:", y_train.shape)
print("Dimensión de y_test:", y_test.shape)

## 13. Escalado de variables

KNN calcula distancias entre observaciones, por lo que es sensible a la escala de las variables. Por eso escalamos las variables predictoras después del train/test split.

Importante: el escalador se ajusta solo con `X_train` y después se aplica a `X_test`, para evitar data leakage.

In [ ]:
# Escalado de variables
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 14. Model Selection: KNN

Usamos `KNeighborsClassifier` porque `Transported` es una variable binaria (`True` / `False`). Por tanto, estamos ante un problema de clasificación.

In [ ]:
# Inicializar el modelo KNN
knn = KNeighborsClassifier(n_neighbors=5)

# Entrenar el modelo
knn.fit(X_train_scaled, y_train);

## 15. Evaluación del modelo

Evaluamos el modelo con `.score()`, que en clasificación devuelve el **accuracy**, es decir, la proporción de predicciones correctas.

In [ ]:
# Evaluar el modelo
train_score = knn.score(X_train_scaled, y_train)
test_score = knn.score(X_test_scaled, y_test)

print("Train score:", train_score)
print("Test score:", test_score)

### Interpretación

El score de entrenamiento muestra cómo funciona el modelo con los datos que ha usado para aprender. El score de test es más importante porque evalúa el rendimiento sobre datos no vistos.

Si el score de train es mucho mayor que el de test, podría haber overfitting. Si ambos son parecidos, el modelo generaliza mejor.

## 16. Probar diferentes valores de `k`

Probamos varios valores de `n_neighbors` para observar cómo cambia el rendimiento del modelo.

In [ ]:
# Probar diferentes valores de k
scores = []

for k in range(1, 16):
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_scaled, y_train)
    score = model.score(X_test_scaled, y_test)
    scores.append({"k": k, "test_score": score})

scores_df = pd.DataFrame(scores)
scores_df

In [ ]:
# Mejor valor de k según el score de test
best_result = scores_df.loc[scores_df["test_score"].idxmax()]
best_result

In [ ]:
# Visualización de resultados por k
plt.figure(figsize=(8, 5))
plt.plot(scores_df["k"], scores_df["test_score"], marker="o")
plt.xlabel("Número de vecinos (k)")
plt.ylabel("Test score")
plt.title("Rendimiento de KNN según el valor de k")
plt.xticks(scores_df["k"])
plt.show()

## 17. Conclusión final

En este lab hemos aplicado técnicas básicas de **feature engineering** antes de entrenar un modelo de Machine Learning.

Pasos principales realizados:

1. Carga y revisión inicial del dataset.
2. Tratamiento de valores nulos con `dropna()`.
3. Transformación de la columna `Cabin` para quedarnos con la cubierta.
4. Eliminación de columnas identificadoras (`PassengerId` y `Name`).
5. Análisis de correlación entre variables numéricas.
6. Test Chi-cuadrado para estudiar asociación entre variables categóricas y `Transported`.
7. Conversión de variables categóricas a dummies.
8. Separación train/test.
9. Escalado de variables para KNN.
10. Entrenamiento y evaluación del modelo.

El objetivo no era solo entrenar un modelo, sino entender cómo la preparación de variables puede afectar al rendimiento y a la interpretación del modelo.